In [1]:
# Install Kaggle
!pip install -q kaggle

# Create Kaggle folder
!mkdir -p ~/.kaggle

# Save your token
!echo "KGAT_a6475a55aafec1db3a2a7482bbba5478" > ~/.kaggle/access_token

# Set permissions
!chmod 600 ~/.kaggle/access_token

# Download dataset
!kaggle datasets download -d aklimarimi/8-facial-expressions-for-yolo

# Unzip dataset
!unzip 8-facial-expressions-for-yolo.zip

Traceback (most recent call last):
  File "/home/jtshi/miniconda3/envs/imgclass/bin/kaggle", line 6, in <module>
    sys.exit(main())
  File "/home/jtshi/miniconda3/envs/imgclass/lib/python3.10/site-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
  File "/home/jtshi/miniconda3/envs/imgclass/lib/python3.10/site-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
  File "/home/jtshi/miniconda3/envs/imgclass/lib/python3.10/site-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
KeyError: 'username'
unzip:  cannot find or open 8-facial-expressions-for-yolo.zip, 8-facial-expressions-for-yolo.zip.zip or 8-facial-expressions-for-yolo.zip.ZIP.


In [2]:
!sudo apt-get -qq install tree

[sudo] password for jtshi: 
sudo: a password is required
^C


In [ ]:
!tree -d

.

0 directories


In [ ]:
import os
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

# ── Constants ─────────────────────────────────────────────────────────────────

CLASSES = [
    'angry', 'contempt', 'disgust', 'fear',
    'happy', 'natural', 'sad', 'sleepy', 'surprised'
]
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224


# ── Dataset ───────────────────────────────────────────────────────────────────

class FacialExpressionDataset(Dataset):
    def __init__(self, base, split, transform=None):
        self.img_dir   = f"{base}/{split}/images"
        self.label_dir = f"{base}/{split}/labels"
        self.transform = transform
        self.samples   = []  # list of (img_path, class_id)

        for lf in sorted(os.listdir(self.label_dir)):
            stem       = os.path.splitext(lf)[0]
            label_path = f"{self.label_dir}/{lf}"

            # find matching image file
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                candidate = f"{self.img_dir}/{stem}{ext}"
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            if img_path is None:
                continue

            # read class id from first annotation line
            with open(label_path) as f:
                lines = f.readlines()
            if not lines:
                continue

            cls_id = int(lines[0].strip().split()[0])
            self.samples.append((img_path, cls_id))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, cls_id = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, cls_id


# ── Transforms ────────────────────────────────────────────────────────────────

train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# ── Class weights ─────────────────────────────────────────────────────────────

def get_class_weights(dataset):
    """Compute inverse frequency weights for each class."""
    counts = Counter(cls_id for _, cls_id in dataset.samples)
    total  = sum(counts.values())
    weights = torch.tensor(
        [total / counts[i] for i in range(NUM_CLASSES)],
        dtype=torch.float
    )
    return weights


# ── Sampler ───────────────────────────────────────────────────────────────────

def get_sampler(dataset, class_weights):
    """Build a WeightedRandomSampler from per-image class weights."""
    sample_weights = torch.tensor(
        [class_weights[cls_id].item() for _, cls_id in dataset.samples],
        dtype=torch.float
    )
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )


# ── DataLoaders ───────────────────────────────────────────────────────────────

def get_dataloaders(base, batch_size=32, num_workers=4):
    """
    Returns train, val, test DataLoaders and class weights for the loss function.

    Args:
        base        : path to dataset root (contains train/, valid/, test/)
        batch_size  : number of samples per batch
        num_workers : parallel workers for data loading

    Returns:
        loaders      : dict with keys 'train', 'val', 'test'
        class_weights: tensor of inverse frequency weights for CrossEntropyLoss
    """
    # datasets
    train_dataset = FacialExpressionDataset(base, "train", transform=train_transforms)
    val_dataset   = FacialExpressionDataset(base, "valid", transform=val_transforms)
    test_dataset  = FacialExpressionDataset(base, "test",  transform=val_transforms)

    # class weights + sampler (train only)
    class_weights = get_class_weights(train_dataset)
    sampler       = get_sampler(train_dataset, class_weights)

    # dataloaders
    loaders = {
        "train": DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,        # replaces shuffle=True
            num_workers=num_workers,
            pin_memory=True         # faster GPU transfer
        ),
        "val": DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True
        ),
        "test": DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True
        )
    }

    return loaders, class_weights


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    base = "./9 Facial Expressions you need"

    loaders, class_weights = get_dataloaders(base, batch_size=32)

    print("Class weights:")
    for cls, w in zip(CLASSES, class_weights):
        print(f"  {cls:12s}: {w:.4f}")

    # sanity check — one batch
    imgs, labels = next(iter(loaders["train"]))
    print(f"\nBatch shape : {imgs.shape}")       # [32, 3, 224, 224]
    print(f"Labels      : {labels}")
    print(f"Pixel range : {imgs.min():.2f} to {imgs.max():.2f}")

FileNotFoundError: [Errno 2] No such file or directory: './9 Facial Expressions you need/train/labels'

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FacialExpressionCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout_rate=0.5):
        super(FacialExpressionCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        # Pooling
        self.pool = nn.MaxPool2d(2, 2)

        # Adaptive pooling to handle variable input sizes
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        # Fully connected layers
        self.fc1 = nn.Linear(256 * 7 * 7, 512)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Conv block 4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        # Adaptive pooling
        x = self.adaptive_pool(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)

        return x

# Quick test
if __name__ == "__main__":
    model = FacialExpressionCNN()
    print(model)
    # Test forward pass
    test_input = torch.randn(1, 3, 224, 224)
    output = model(test_input)
    print(f"\nOutput shape: {output.shape}")

In [ ]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

def train_model(model, loaders, class_weights, config, device='cuda'):
    """
    Train model with given hyperparameters

    Args:
        config: dict containing:
            - learning_rate
            - weight_decay
            - epochs
            - dropout_rate
            - optimizer_type ('adam' or 'sgd')
            - scheduler_patience
    """

    model = model.to(device)
    class_weights = class_weights.to(device)

    # Loss function with class weights
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Optimizer
    if config['optimizer_type'] == 'adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
    else:  # sgd
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config['learning_rate'],
            momentum=0.9,
            weight_decay=config['weight_decay']
        )

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=config['scheduler_patience'],
        factor=0.5, verbose=True
    )

    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    best_val_acc = 0.0

    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, labels in tqdm(loaders['train'], desc=f'Epoch {epoch+1}/{config["epochs"]} [Train]'):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100.0 * train_correct / train_total
        avg_train_loss = train_loss / len(loaders['train'])

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in tqdm(loaders['val'], desc=f'Epoch {epoch+1}/{config["epochs"]} [Val]'):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100.0 * val_correct / val_total
        avg_val_loss = val_loss / len(loaders['val'])

        # Update scheduler
        scheduler.step(avg_val_loss)

        # Save history
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_model_{config["name"]}.pth')

        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')

    return history, best_val_acc

In [ ]:
def evaluate_model(model, loaders, device='cuda'):
    """Evaluate model on test set and return metrics"""
    model.eval()
    model.to(device)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(loaders['test'], desc='Testing'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import classification_report, confusion_matrix

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=CLASSES))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar()
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    # Add text annotations
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')

    plt.xticks(range(len(CLASSES)), CLASSES, rotation=45)
    plt.yticks(range(len(CLASSES)), CLASSES)
    plt.tight_layout()
    plt.show()

    test_acc = 100.0 * (np.array(all_preds) == np.array(all_labels)).sum() / len(all_labels)
    return test_acc

def plot_training_history(history, config_name):
    """Plot training curves"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{config_name} - Loss Curves')
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{config_name} - Accuracy Curves')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import pickle
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

# ── RTX 4050 optimisations (run once at startup) ──────────────────────────────
torch.backends.cudnn.benchmark = True          # auto-tune kernels for fixed input sizes


def train_model(model, loaders, class_weights, config, device=None, checkpoint_freq=2):
    """
    Train model with given hyperparameters and checkpoint saving.
    Optimised for RTX 4050 + Ryzen 7 + 32 GB RAM.

    Args:
        config: dict containing:
            - name: configuration name
            - learning_rate
            - weight_decay
            - epochs
            - dropout_rate
            - optimizer_type ('adam' or 'sgd')
            - scheduler_patience
        checkpoint_freq: save checkpoint every N epochs
    """

    # ── Auto-detect GPU ───────────────────────────────────────────────────────
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"🖥️  Using device: {device}")

    model = model.to(device)
    class_weights = class_weights.to(device)

    # ── Compile model (PyTorch 2.0+, free speed boost) ───────────────────────
    if hasattr(torch, 'compile'):
        print("⚡ Compiling model with torch.compile …")
        model = torch.compile(model)

    # ── Loss & optimiser ──────────────────────────────────────────────────────
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    if config['optimizer_type'] == 'adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
    else:  # sgd
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config['learning_rate'],
            momentum=0.9,
            weight_decay=config['weight_decay']
        )

    # ── LR scheduler ─────────────────────────────────────────────────────────
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=config['scheduler_patience'], factor=0.5
    )

    # ── Mixed-precision scaler (FP16 on RTX 4050) ─────────────────────────────
    # Roughly doubles throughput and halves VRAM usage on Ampere/Ada GPUs.
    use_amp = (device == 'cuda')
    scaler = GradScaler(enabled=use_amp)
    if use_amp:
        print("🔥 Mixed-precision (FP16) training enabled")

    # ── Checkpoint paths ──────────────────────────────────────────────────────
    checkpoint_path = f'checkpoint_{config["name"]}.pth'
    history_path    = f'history_{config["name"]}.pkl'

    # ── Try to resume from checkpoint ─────────────────────────────────────────
    start_epoch  = 0
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    if os.path.exists(checkpoint_path):
        print("🔄 Found existing checkpoint! Resuming training …")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])

        # Restore scaler state if it was saved
        if 'scaler_state' in checkpoint and use_amp:
            scaler.load_state_dict(checkpoint['scaler_state'])

        start_epoch  = checkpoint['epoch'] + 1
        best_val_acc = checkpoint['best_val_acc']
        history      = checkpoint['history']
        current_lr   = optimizer.param_groups[0]['lr']

        print(f"   Resuming from epoch {start_epoch}/{config['epochs']}")
        print(f"   Previous best validation accuracy: {best_val_acc:.2f}%")
        print(f"   Current learning rate: {current_lr:.6f}")
        print(f"   History length: {len(history['train_loss'])} epochs")

        if start_epoch >= config['epochs']:
            print("   ✓ Training already completed for this config!")
            return history, best_val_acc
    else:
        print(f"📝 Starting fresh training for {config['name']}")

    # Load separate history file if checkpoint history is empty
    if os.path.exists(history_path) and not history['train_loss']:
        print(f"   Loading additional history from {history_path}")
        with open(history_path, 'rb') as f:
            saved_history = pickle.load(f)
            if len(saved_history['train_loss']) > len(history['train_loss']):
                history = saved_history

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, config['epochs']):

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        train_loader = tqdm(loaders['train'], desc=f'Epoch {epoch+1}/{config["epochs"]} [Train]')
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # set_to_none=True is faster than zeroing gradients
            optimizer.zero_grad(set_to_none=True)

            # Forward pass under autocast (FP16 where safe, FP32 elsewhere)
            with autocast(enabled=use_amp):
                outputs = model(inputs)
                loss    = criterion(outputs, labels)

            # Scaled backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss    += loss.item()
            _, predicted   = outputs.max(1)
            train_total   += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            train_loader.set_postfix({'loss': loss.item()})

        train_acc      = 100.0 * train_correct / train_total
        avg_train_loss = train_loss / len(loaders['train'])

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            val_loader = tqdm(loaders['val'], desc=f'Epoch {epoch+1}/{config["epochs"]} [Val]')
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                with autocast(enabled=use_amp):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)

                val_loss    += loss.item()
                _, predicted = outputs.max(1)
                val_total   += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc      = 100.0 * val_correct / val_total
        avg_val_loss = val_loss / len(loaders['val'])

        # ── LR scheduler step ─────────────────────────────────────────────────
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f"   📉 LR reduced: {old_lr:.6f} → {new_lr:.6f}")

        # ── Record history ────────────────────────────────────────────────────
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        # ── Save best model ───────────────────────────────────────────────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_model_{config["name"]}.pth')
            print(f"   ✨ New best model! Val accuracy: {val_acc:.2f}%")

        # ── Save checkpoint ───────────────────────────────────────────────────
        if (epoch + 1) % checkpoint_freq == 0 or epoch == config['epochs'] - 1:
            checkpoint = {
                'epoch':           epoch,
                'model_state':     model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state':    scaler.state_dict(),   # save AMP scaler state
                'best_val_acc':    best_val_acc,
                'history':         history,
                'config':          config,
            }
            torch.save(checkpoint, checkpoint_path)

            with open(history_path, 'wb') as f:
                pickle.dump(history, f)

            print(f"   💾 Checkpoint saved at epoch {epoch+1}")

        print(
            f'Epoch {epoch+1}: '
            f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
            f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%, '
            f'LR: {new_lr:.6f}'
        )

    # ── Cleanup on successful completion ──────────────────────────────────────
    print(f"\n🎉 Training completed for {config['name']}!")
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print("   Removed checkpoint file (training complete)")

    return history, best_val_acc

In [ ]:
# Test single configuration with error handling
test_config =     {
        'name': 'config_5_sgd_optimizer',
        'learning_rate': 0.01,
        'weight_decay': 0.0001,
        'epochs': 10,
        'dropout_rate': 0.5,
        'optimizer_type': 'sgd',
        'scheduler_patience': 3
    }

print(f"Using device: {device}")
print(f"Test config: {test_config['name']}")

try:
    # Create model
    print("\n1. Creating model...")
    model = FacialExpressionCNN(dropout_rate=test_config['dropout_rate'])
    print(f"   ✓ Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

    # Train model
    print("\n2. Starting training...")
    history, best_val_acc = train_model(model, loaders, class_weights, test_config, device)
    print(f"   ✓ Training completed. Best val acc: {best_val_acc:.2f}%")

    # Evaluate on test set
    print("\n3. Evaluating on test set...")
    test_acc = evaluate_model(model, loaders, device)
    print(f"   ✓ Test accuracy: {test_acc:.2f}%")

    # Store result
    results = {test_config['name']: {
        'history': history,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'config': test_config
    }}

except Exception as e:
    print(f"\n✗ ERROR: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Optimized hyperparameter configurations for faster tuning
hyperparameter_configs = [
    {
        'name': 'config_1_baseline',
        'learning_rate': 0.001,
        'weight_decay': 0.0001,
        'epochs': 10,  # 10 epochs for quicker tuning
        'dropout_rate': 0.5,
        'optimizer_type': 'adam',
        'scheduler_patience': 3
    },
    {
        'name': 'config_2_lr_higher',
        'learning_rate': 0.01,
        'weight_decay': 0.0001,
        'epochs': 10,
        'dropout_rate': 0.5,
        'optimizer_type': 'adam',
        'scheduler_patience': 3
    },
    {
        'name': 'config_3_lower_weight_decay',
        'learning_rate': 0.001,
        'weight_decay': 0.00001,
        'epochs': 10,
        'dropout_rate': 0.5,
        'optimizer_type': 'adam',
        'scheduler_patience': 3
    },
    {
        'name': 'config_4_higher_dropout',
        'learning_rate': 0.001,
        'weight_decay': 0.0001,
        'epochs': 10,
        'dropout_rate': 0.7,
        'optimizer_type': 'adam',
        'scheduler_patience': 3
    },
    {
        'name': 'config_5_sgd_optimizer',
        'learning_rate': 0.01,
        'weight_decay': 0.0001,
        'epochs': 10,
        'dropout_rate': 0.5,
        'optimizer_type': 'sgd',
        'scheduler_patience': 3
    }
]

# Run experiments with progress tracking
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Training samples: {len(loaders['train'].dataset)}")
print(f"Validation samples: {len(loaders['val'].dataset)}")
print(f"Test samples: {len(loaders['test'].dataset)}")

results = {}

for i, config in enumerate(hyperparameter_configs, 1):
    print("\n" + "="*60)
    print(f"Experiment {i}/{len(hyperparameter_configs)}: {config['name']}")
    print("="*60)
    print(f"Hyperparameters:")
    print(f"  - Learning rate: {config['learning_rate']}")
    print(f"  - Weight decay: {config['weight_decay']}")
    print(f"  - Dropout rate: {config['dropout_rate']}")
    print(f"  - Optimizer: {config['optimizer_type']}")
    print(f"  - Epochs: {config['epochs']}")

    try:
        # Create model with current config
        model = FacialExpressionCNN(dropout_rate=config['dropout_rate'])
        print(f"\n✓ Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

        # Train model
        print(f"\n▶ Starting training...")
        history, best_val_acc = train_model(model, loaders, class_weights, config, device)

        # Evaluate on test set
        print(f"\n▶ Evaluating on test set...")
        test_acc = evaluate_model(model, loaders, device)

        # Store results
        results[config['name']] = {
            'history': history,
            'best_val_acc': best_val_acc,
            'test_acc': test_acc,
            'config': config
        }

        # Plot training curves
        plot_training_history(history, config['name'])

        print(f"\n✅ {config['name']} COMPLETED:")
        print(f"   Best Validation Accuracy: {best_val_acc:.2f}%")
        print(f"   Test Accuracy: {test_acc:.2f}%")

    except Exception as e:
        print(f"\n❌ {config['name']} FAILED:")
        print(f"   Error: {e}")
        import traceback
        traceback.print_exc()
        print(f"\n⚠ Continuing with next configuration...")
        continue

# Display comparison of successful runs
if results:
    print("\n" + "="*60)
    print("HYPERPARAMETER TUNING RESULTS SUMMARY")
    print("="*60)

    # Create comparison dataframe
    comparison_data = []
    for name, result in results.items():
        comparison_data.append({
            'Configuration': name,
            'LR': result['config']['learning_rate'],
            'Weight Decay': result['config']['weight_decay'],
            'Dropout': result['config']['dropout_rate'],
            'Optimizer': result['config']['optimizer_type'],
            'Best Val Acc (%)': f"{result['best_val_acc']:.2f}",
            'Test Acc (%)': f"{result['test_acc']:.2f}"
        })

    import pandas as pd
    comparison_df = pd.DataFrame(comparison_data)
    print("\n", comparison_df.to_string(index=False))

    # Find best configuration by test accuracy
    best_config_name = max(results.items(), key=lambda x: x[1]['test_acc'])[0]
    best_result = results[best_config_name]

    print(f"\n🏆 BEST CONFIGURATION: {best_config_name}")
    print(f"   Test Accuracy: {best_result['test_acc']:.2f}%")
    print(f"   Validation Accuracy: {best_result['best_val_acc']:.2f}%")
    print(f"   Hyperparameters:")
    for key, value in best_result['config'].items():
        if key != 'name':
            print(f"     - {key}: {value}")

    # Plot comparison bar chart
    plt.figure(figsize=(12, 6))
    config_names = list(results.keys())
    test_accs = [results[name]['test_acc'] for name in config_names]
    val_accs = [results[name]['best_val_acc'] for name in config_names]

    x = np.arange(len(config_names))
    width = 0.35

    bars1 = plt.bar(x - width/2, val_accs, width, label='Validation Accuracy', color='skyblue', alpha=0.8)
    bars2 = plt.bar(x + width/2, test_accs, width, label='Test Accuracy', color='lightcoral', alpha=0.8)

    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

    for bar in bars2:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

    plt.xlabel('Configurations', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title('Hyperparameter Comparison for Facial Expression Recognition', fontsize=14, fontweight='bold')
    plt.xticks(x, config_names, rotation=45, ha='right')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # Save results for report
    import json
    from datetime import datetime

    summary = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'best_configuration': best_config_name,
        'best_test_accuracy': float(best_result['test_acc']),
        'best_validation_accuracy': float(best_result['best_val_acc']),
        'dataset_info': {
            'train_size': len(loaders['train'].dataset),
            'val_size': len(loaders['val'].dataset),
            'test_size': len(loaders['test'].dataset),
            'num_classes': NUM_CLASSES,
            'classes': CLASSES
        },
        'all_results': {
            name: {
                'test_accuracy': float(res['test_acc']),
                'best_validation_accuracy': float(res['best_val_acc']),
                'hyperparameters': {k: v for k, v in res['config'].items() if k != 'history'}
            }
            for name, res in results.items()
        }
    }

    with open('hyperparameter_tuning_results.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print("\n📊 Results saved to 'hyperparameter_tuning_results.json'")

else:
    print("\n❌ No configurations completed successfully. Please check the errors above.")

In [ ]:
# Save all results to files for your report
import json
from datetime import datetime

# Save summary
summary = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'best_config': best_config[0],
    'best_test_accuracy': float(best_config[1]['test_acc']),
    'all_results': {
        name: {
            'test_accuracy': float(res['test_acc']),
            'best_val_accuracy': float(res['best_val_acc']),
            'hyperparameters': res['config']
        }
        for name, res in results.items()
    }
}

with open('hyperparameter_tuning_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Results saved to 'hyperparameter_tuning_results.json'")

# Generate markdown table for report
markdown_table = "| Config | LR | Weight Decay | Dropout | Optimizer | Val Acc (%) | Test Acc (%) |\n"
markdown_table += "|--------|----|--------------|---------|-----------|-------------|--------------|\n"

for name, res in results.items():
    cfg = res['config']
    markdown_table += f"| {name} | {cfg['learning_rate']} | {cfg['weight_decay']} | {cfg['dropout_rate']} | "
    markdown_table += f"{cfg['optimizer_type']} | {res['best_val_acc']:.2f} | {res['test_acc']:.2f} |\n"

with open('results_table.md', 'w') as f:
    f.write(markdown_table)

print("\nMarkdown table saved to 'results_table.md'")
print("\n" + markdown_table)